# Modélisation du churn — Spark MLlib (LECTURE sur Gold)

Itération sur le modèle ML. Lecture des features depuis `gold/churn_features`,
écriture des prédictions dans `gold/churn_predictions`.

In [ ]:
import sys; sys.path.insert(0, '..')
from src import config
spark = config.build_spark(app_name='modeling')

In [ ]:
df = spark.read.parquet(config.s3_path('gold_feat'))
df = df.na.drop(subset=['churn'])
print('features :', df.count())

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier

num = ['tenure', 'monthly_charges', 'total_charges', 'num_services']
num = [c for c in num if c in df.columns]
cat = ['contract', 'tenure_bucket']
cat = [c for c in cat if c in df.columns]

idx = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat]
ohe = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_oh') for c in cat]
assembler = VectorAssembler(inputCols=num + [c+'_oh' for c in cat], outputCol='features')
gbt = GBTClassifier(labelCol='churn', featuresCol='features', maxIter=30)
pipeline = Pipeline(stages=idx + ohe + [assembler, gbt])

In [ ]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
modelfit = pipeline.fit(train)
pred = modelfit.transform(test)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
auc = BinaryClassificationEvaluator(labelCol='churn').evaluate(pred)
print('AUC :', round(auc, 4))

In [ ]:
# Écriture des prédictions dans Gold
from pyspark.sql import functions as F
out = pred.select('*', F.col('prediction').alias('churn_pred'))
out.write.mode('overwrite').parquet(config.s3_path('gold_pred'))
# Modèle sérialisé
modelfit.write().overwrite().save(config.s3_path('models', 'gbt_churn'))